# <font style="font-family:roboto;color:#455e6c"> Parametrising a Machine Learning Interatomic Potential </font>  

<div class="admonition note" name="html-admonition" style="background:#e3f2fd; padding: 10px">
<font style="font-family:roboto;color:#455e6c"> <b> DPG Tutorial: Automated Workflows and Machine Learning for Materials Science Simulations </b> </font> </br>
<font style="font-family:roboto;color:#455e6c"> 8 March 2026 </font> </br> </br>
Sriram Anand, Prabhath Chilakalapudi, Haitham Gaafer, Marvin Poul, Jan Janssen, Jörg Neugebauer </br>
<i> Max Planck Institute for Sustainable Materials </i></br>
</br>
Sarath Menon, Minaam Qamar, Ralf Drautz </br>
<i> Ruhr-Universität Bochum </i></br>
</br>
Tilmann Hickel </br>
<i> Bundesanstalt für Materialforschung und -prüfung </i></br>
</div>

### References

- [Lysogorskiy, Y. et al. Performant implementation of the atomic cluster expansion (PACE) and application to copper and silicon. npj Comput Mater 7, 97 (2021)](http://www.nature.com/articles/s41524-021-00559-9)

In this notebook we fit an [Atomic Cluster Expansion](https://doi.org/10.1103/PhysRevB.99.014104) interatomic potential using the [pacemaker](https://www.nature.com/articles/s41524-021-00559-9) software.

## <font style="font-family:roboto;color:#455e6c"> Loading and Analyzing the dataset </font> 

Recalling the workflow, we are in the first essential step of loading the training dataset

<div style="text-align: center;">
    <img src="img/highlighted_workflow.png" alt="Fitting workflow" width="70%">
</div>

As a first step, we load the dataset by specifying the `file_path'. This dataset has been generated for the CaMg system using the ASSYST approach discussed in the previous section [02_assyst.ipynb](02_assyst.ipynb).

## <font style="font-family:roboto;color:#455e6c"> Split the dataset into training and test </font> 

In a second step, we split the dataset into training and testing datasets.

This is done by choosing the percentage used for the training dataset through the `training_frac` parameter, where `training_frac = 0.5` means we use 50% for training and 50% for testing.

The histogram of the total energies of all atomic structures in the dataset is plotted using the `PlotEnergyHistogram` node. Similarly, using the `PlotForcesHistogram` Node, we can plot the forces histogram

## <font style="font-family:roboto;color:#455e6c"> Define and specify the configuration of the ACE potential </font> 

Following the `PACEmaker` notation, we create a file similar to `input.yaml` (see the `PACEmaker` [documentation](https://pacemaker.readthedocs.io/en/latest/) for more details).

```
fit:
  fit_cycles: 1
  loss: {L1_coeffs: 1e-08, L2_coeffs: 1e-08, kappa: 0.08, w0_rad: 0, w1_rad: 0, w2_rad: 0}
  maxiter: 1000
  optimizer: BFGS
  trainable_parameters: ALL
  weighting: {DE: 1.0, DElow: 1.0, DEup: 10.0, DF: 1.0, DFup: 50.0, energy: convex_hull,
    nfit: 20000, reftype: all, seed: 42, type: EnergyBasedWeightingPolicy, wlow: 0.95}
.
.
.
potential:
  bonds:
    ALL:
      dcut: 0.01
      radbase: SBessel
      radparameters: [5.25]
      rcut: 6.0
  elements: [Mg, Ca]
  embeddings:
    ALL:
      fs_parameters: [1, 1, 1, 0.5]
      ndensity: 2
      npot: FinnisSinclairShiftedScaled
  functions:
    number_of_functions_per_element = 300
    ALL:
      lmax_by_orders: [15, 6, 2, 1]
      nradmax_by_orders: [0, 6, 3, 1]

### 1. Embeddings
specify how the atomic energy $E_i$ depends on the ACE properties/densities $\varphi$. The most approximate approach, but the most efficient for potential fitting, is the linear expansion $E_i = \varphi$. Non-linear expansions, e.g. including the square root, provide more flexibility and accuracy of the final potential, but require significantly more computational resources for fitting.

Embeddings for `ALL` species: 
- non-linear `FinnisSinclairShiftedScaled`
- 2 densities
- fs_parameters': [1, 1, 1, 0.5]:
$$E_i = 1.0 * \varphi(1)^1 + 1.0 * \varphi(2)^{0.5} = \varphi^{(1)} + \sqrt{\varphi^{(2)}} $$

### 2. Radial functions

Radial functions are defined by orthogonal polynomals. Examples are:
* (a) Exponentially-scaled Chebyshev polynomials (λ = 5.25)
* (b) Power-law scaled Chebyshev polynomials (λ = 2.0)
* (c) Simplified spherical Bessel functions

<div style="text-align: center;">
  <img src="img/radial-functions-low.png" width="40%">
</div>


Radial functions specification have to be provided for `ALL` species pairs (i.e. Al-Al, Al-Li, Li-Al, Li-Li):

* based on the Simplified Bessel function
* and cutoff, e.g. $r_c=6.0$

#### 3. B-basis functions

B-basis functions  specifications for `ALL` species type interactions, i.e. Al-Al block:
* maximum order = 4, i.e. body-order 5 (1 central atom + 4 neighbour  densities)
* nradmax_by_orders: 15, 3, 2, 1
* lmax_by_orders: 0, 3, 2, 1

For simplicity, the main inputs that we will consider for the potential configurations are:

- `number_of_functions_per_element`: specifies how many functions will be provided in the potential
- `rcut`: specifies what the cutoff radius is

## <font style="font-family:roboto;color:#455e6c"> Linear fitting </font>

Finally, we run our fit with the thus defined `potential_config` and then save the potential files inside a new folder.

**Note:** Setting `verbose = True` will show all the details of building the design matrices.

## <font style="font-family:roboto;color:#455e6c"> Workflow For Fitting and Validating the Potential </font>

<div class="admonition note" name="html-admonition" style="background: #FFEDD1; padding: 10px">
<p class="title"><b>Task</b></p>

Create a workflow to fit a linear ACE potential, then perform a level 1 validation of the potential. Consider the following steps,
1. We need to get the `RunLinearFit` node and also `ParameterizePotentialConfig` node.
2. Use the fitted potential with `PredictEnergiesAndForces` node for both training and testing datasets.
3. Add fitting plotting nodes such as `PlotEnergyFittingCurve` or `PlotForcesFittingCurve` to see how accurate our fitted potentials are with respect to predicting `energies` and `forces`.

**Hint**: You can find the relevant nodes to perform the fitting in `atomistic` -> `fitting` -> `ace` and for plotting in `atomistic` -> `fitting` -> `plot`

Exercise: change the `number_of_functions_per_element` to a higher value (i.e., 50) and check the fitting curves. Did the fit get better or worse? why?

**Note:** You can save the potential under a new name using the `filename` input in `save_potential` node.
</div>

<div style="text-align: center;">
    <img src="img/validation_schematic.png" width="60%">
</div>

In [1]:
from core import Workflow
from core.gui import PyironFlow, GUILayout

In [2]:
from pyiron_nodes.atomistic.fitting.dataset import ReadPickledDatasetAsDataframe, SplitTrainingAndTesting
from pyiron_nodes.atomistic.fitting.ace import ParameterizePotentialConfig, RunLinearFit, PredictEnergiesAndForces
from pyiron_nodes.atomistic.fitting.plot import PlotEnergyHistogram, PlotForcesHistogram, PlotEnergyFittingCurve, PlotForcesFittingCurve

In [3]:
def make_linearfit(
    workflow_name: str,
    file_path: str = "data/mgca.pckl.tgz",
    compression: str | None = None,
    training_frac: float | int = 0.5,
    number_of_functions_per_element: int | None = 10,
    rcut: float | int = 6.0,
):

    wf = Workflow(workflow_name)
    # Workflow connections
    wf.LoadDataset = ReadPickledDatasetAsDataframe(
        file_path=file_path, compression=compression
    )
    wf.PlotEnergyHistogram = PlotEnergyHistogram(
        wf.LoadDataset
    )
    wf.PlotForcesHistogram = PlotForcesHistogram(
        wf.LoadDataset
    )
    wf.SplitDataset = SplitTrainingAndTesting(
        data_df=wf.LoadDataset.outputs.df, training_frac=training_frac
    )
    wf.ParameterizePotentialConfig = ParameterizePotentialConfig(
        number_of_functions_per_element=number_of_functions_per_element, rcut=rcut
    )
    
    wf.RunLinearFit = RunLinearFit(
        potential_config=wf.ParameterizePotentialConfig,
        df_train=wf.SplitDataset.outputs.df_training,
        df_test=wf.SplitDataset.outputs.df_testing,
        verbose=False,
    )
    wf.PredictEnergiesAndForces = PredictEnergiesAndForces(
        potential_file_path=wf.RunLinearFit.outputs.potential_file_path,
        df_train=wf.SplitDataset.outputs.df_training,
        df_test=wf.SplitDataset.outputs.df_testing,
    )
    wf.PlotEnergyFittingCurve = PlotEnergyFittingCurve(wf.PredictEnergiesAndForces)
    wf.PlotForcesFittingCurve = PlotForcesFittingCurve(wf.PredictEnergiesAndForces)
    return wf

In [4]:
wf = make_linearfit("RunLinearFit_solution")

In [5]:
workflow_path = "pyiron_core_data/workflows/"
pf = PyironFlow([wf], nodes_path="pyiron_nodes", workflow_path = workflow_path)
pf.gui

<div style="text-align: center;">
    <img src="img/fit_example.png" width="100%">
</div>

<img src="img/logo_roll.png" width="1200">